# Phishing URL Detection — Project Notebook

This notebook walks through a complete machine learning pipeline to classify URLs as **phishing** or **legitimate**.

**Objectives:**
- Use three classifiers: Random Forest, KNN, Decision Tree
- Compare their performance on accuracy, precision, recall, F1-score
- Build a reusable function to test any URL

**Data:**
- `Training.parquet`: 7,658 rows × 89 columns
- `Testing.parquet`: 3,772 rows × 89 columns
- Combined for feature engineering: **11,430 rows**
- **Feature selection:** From the 89 original columns, we keep only the **12 features + 1 target** defined in the data dictionary (DATAF_e0224020.pdf):

  **Features (12):**
  1. `URLlength` — total URL length
  2. `DomainLength` — hostname/domain length
  3. `IsDomainIP` — binary (1 if domain is an IP address)
  4. `TLD` — top-level domain (categorical, label encoded)
  5. `NoOfSubDomain` — number of subdomains
  6. `NoOfLettersInURL` — count of alphabetic characters
  7. `NoOfDigitsInURL` — count of numeric characters
  8. `NoOfEqualsInURL` — count of `=` symbols
  9. `NoOfQMarkInURL` — count of `?` symbols
  10. `NoOfAmpersandInURL` — count of `&` symbols
  11. `SpecialCharCount` — count of non-alphanumeric, non-dot characters
  12. `Entropy` — Shannon entropy of the URL string

  **Target (1):**
  - `Label` — 0 = legitimate, 1 = phishing

- Train/test split happens *after* combining (80/20 stratified)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, math
from urllib.parse import urlparse
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Load Data & Build the 14 Variables

**Next cell:**
- Load both parquet files and combine into one DataFrame (11,430 rows)
- Define helper functions to extract TLD, count letters/digits/special chars, compute Shannon entropy
- Build 14 columns: 12 URL-derived features + raw URL + target Label
- Subdomain count comes from existing `nb_subdomains` column
- Print shape and preview first rows

In [ ]:
raw = pd.concat([pd.read_parquet('Training.parquet'),
                 pd.read_parquet('Testing.parquet')],
                ignore_index=True)

def get_tld(url):
    try:
        host = urlparse(url).netloc
        return host.split('.')[-1] if '.' in host else 'none'
    except Exception:
        return 'none'

def count_letters(url):   return len(re.findall(r'[A-Za-z]', url))
def count_digits(url):    return len(re.findall(r'\d', url))

def special_char_count(url):
    return len(re.findall(r'[^A-Za-z0-9.]', url))

def shannon_entropy(text):
    if not text:
        return 0.0
    freq = Counter(text)
    n = len(text)
    return -sum((c/n) * math.log2(c/n) for c in freq.values())

df = pd.DataFrame()
df['URL']              = raw['url']
df['URLlength']        = raw['length_url']
df['DomainLength']     = raw['length_hostname']
df['IsDomainIP']       = raw['ip']
df['TLD']              = raw['url'].apply(get_tld)
df['NoOfSubDomain']    = raw['nb_subdomains']
df['NoOfLettersInURL'] = raw['url'].apply(count_letters)
df['NoOfDigitsInURL']  = raw['url'].apply(count_digits)
df['NoOfEqualsInURL']  = raw['nb_eq']
df['NoOfQMarkInURL']   = raw['nb_qm']
df['NoOfAmpersandInURL'] = raw['nb_and']
df['SpecialCharCount'] = raw['url'].apply(special_char_count)
df['Entropy']          = raw['url'].apply(shannon_entropy)
df['Label']            = raw['status']

print('Shape:', df.shape)
df.head()

## 2. Clean Data & Encode Categoricals

**Next cell:**
- Verify no missing values exist
- Encode `TLD` (categorical) and `Label` (target) into integers using `LabelEncoder`
- Drop raw `URL` column — we've extracted everything needed
- Define final 12 feature columns for input matrix `X`
- Set encoded `Label` as target vector `y`
- Print label mapping and class balance

In [ ]:
print('Missing values:\n', df.isnull().sum())

le_tld = LabelEncoder()
df['TLD'] = le_tld.fit_transform(df['TLD'])

le_y = LabelEncoder()
df['Label'] = le_y.fit_transform(df['Label'])

print('\nLabel mapping:', dict(zip(le_y.classes_, le_y.transform(le_y.classes_))))
print('\nClass balance:\n', df['Label'].value_counts())

feature_cols = ['URLlength','DomainLength','IsDomainIP','TLD','NoOfSubDomain',
                'NoOfLettersInURL','NoOfDigitsInURL','NoOfEqualsInURL',
                'NoOfQMarkInURL','NoOfAmpersandInURL','SpecialCharCount','Entropy']

X = df[feature_cols]
y = df['Label']
X

## 3. Exploratory Data Analysis (Optional)

**Next cell:**
- Plot class distribution (should be ~50/50 balanced)
- Boxplots: URL length, Entropy, SpecialCharCount split by class
- Correlation heatmap of all 12 input features
- Helps spot class separation and feature redundancy before modeling

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.countplot(x='Label', data=df, ax=axes[0,0], palette='Set2')
axes[0,0].set_title('Class Distribution (0=Legitimate, 1=Phishing)')

sns.boxplot(x='Label', y='URLlength', data=df, ax=axes[0,1], palette='Set2')
axes[0,1].set_title('URL Length by Class')

sns.boxplot(x='Label', y='Entropy', data=df, ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Entropy by Class')

sns.boxplot(x='Label', y='SpecialCharCount', data=df, ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Special Character Count by Class')

plt.tight_layout()
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(X.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation of the 12 Input Variables')
plt.show()

## 4. Split Data & Scale for KNN

**Next cell:**
- Stratified 80/20 train/test split (preserves class balance)
- Fit `StandardScaler` on training data only (no leakage)
- Apply scaler to both train and test sets
- KNN needs scaled features; Random Forest and Decision Tree use unscaled

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Train:', X_train.shape, ' Test:', X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 5. Select & Train Three Models

**Next cell:**
- Instantiate Random Forest (100 trees, parallel jobs)
- Instantiate KNN (k=5, distance-weighted voting)
- Instantiate Decision Tree (reproducible seed)
- Fit all three on training data
- KNN uses scaled features; RF and DT use raw features
- Generate predictions on test set for each model

In [ ]:
rf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
dt  = DecisionTreeClassifier(random_state=42)

rf.fit(X_train, y_train)
knn.fit(X_train_scaled, y_train)
dt.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_pred_knn = knn.predict(X_test_scaled)
y_pred_dt  = dt.predict(X_test)

print('Models trained.')

## 6. Evaluate Each Model

**Next cell:**
- Compute Accuracy, Precision, Recall, F1-Score for all three models
- Build comparison DataFrame and print rounded results
- Print detailed classification report per model (per-class metrics)
- See how each handles phishing vs. legitimate separately

In [ ]:
models   = ['Random Forest', 'KNN', 'Decision Tree']
preds    = {'Random Forest': y_pred_rf, 'KNN': y_pred_knn, 'Decision Tree': y_pred_dt}

rows = []
for m in models:
    rows.append({
        'Model':     m,
        'Accuracy':  accuracy_score(y_test, preds[m]),
        'Precision': precision_score(y_test, preds[m]),
        'Recall':    recall_score(y_test, preds[m]),
        'F1-Score':  f1_score(y_test, preds[m])
    })

results = pd.DataFrame(rows).set_index('Model')
print(results.round(4))
print()

for m in models:
    print(f'--- {m} ---')
    print(classification_report(y_test, preds[m],
                                target_names=['Legitimate','Phishing']))

## 7. Confusion Matrices

**Next cell:**
- Plot three heatmaps side by side (one per model)
- Annotate with true/false positive/negative counts
- Easy visual: which model misses most phishing (false negatives)
- Which incorrectly flags legitimate (false positives)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(axes, models):
    cm = confusion_matrix(y_test, preds[m])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legitimate','Phishing'],
                yticklabels=['Legitimate','Phishing'])
    ax.set_title(f'{m}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 8. Compare Models & Pick the Best

**Next cell:**
- Grouped bar chart: Accuracy, Precision, Recall, F1-Score for all models
- Identify highest F1-Score → best balance of precision & recall
- Print winning model name and its F1-Score

In [ ]:
results[['Accuracy','Precision','Recall','F1-Score']].plot(
    kind='bar', figsize=(10,5), colormap='viridis')
plt.title('Model Comparison')
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.ylabel('Score')
plt.legend(loc='lower right')
plt.show()

best = results['F1-Score'].idxmax()
print(f'Best model (highest F1-Score): {best}  ->  {results.loc[best,"F1-Score"]:.4f}')

## 9. Feature Importance (Random Forest)

**Next cell:**
- Extract feature importances from trained Random Forest
- Sort and plot horizontal bar chart (12 features ranked)
- Reveals which URL characteristics drive phishing detection

In [ ]:
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()
imp.plot(kind='barh', figsize=(8,5), color='teal')
plt.title('Random Forest - Feature Importance')
plt.xlabel('Importance')
plt.show()

## 10. Test a Custom URL

**Next cell:**
- `predict_url()` function: engineers 12 features from any raw URL
- Applies same TLD encoding and KNN scaling
- Runs all three models, returns majority-vote verdict
- Prints final result + each model's individual call
- Edit `url` variable at top and re-run to test your own links

In [ ]:
url = "https://www.google.com"

def predict_url(url):
    netloc = urlparse(url).netloc
    feat = {
        'URLlength':        len(url),
        'DomainLength':     len(netloc),
        'IsDomainIP':       int(bool(re.match(r'^(\d{1,3}\.){3}\d{1,3}$', netloc.split(':')[0]))),
        'TLD':              le_tld.transform([get_tld(url)])[0] if get_tld(url) in le_tld.classes_ else -1,
        'NoOfSubDomain':    max(0, len(netloc.split('.')) - 2),
        'NoOfLettersInURL': count_letters(url),
        'NoOfDigitsInURL':  count_digits(url),
        'NoOfEqualsInURL':  url.count('='),
        'NoOfQMarkInURL':   url.count('?'),
        'NoOfAmpersandInURL': url.count('&'),
        'SpecialCharCount': special_char_count(url),
        'Entropy':          shannon_entropy(url),
    }
    x = pd.DataFrame([feat])[feature_cols]

    rf_pred  = rf.predict(x)[0]
    knn_pred = knn.predict(scaler.transform(x))[0]
    dt_pred  = dt.predict(x)[0]

    votes = [rf_pred, knn_pred, dt_pred]
    final = 1 if votes.count(1) > 1 else 0

    return ('PHISHING' if final else 'LEGITIMATE'), \
           {'Random Forest': 'Phishing' if rf_pred else 'Legitimate',
            'KNN':           'Phishing' if knn_pred else 'Legitimate',
            'Decision Tree': 'Phishing' if dt_pred else 'Legitimate'}

if url:
    result, detail = predict_url(url)
    print('URL  :', url)
    print('Result:', result)
    for k, v in detail.items():
        print(f'  {k:15s}: {v}')